In [1]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Carrega a tabela que você já consolidou no Bloco 2
caminho_csv = "../data/corpus_favela_extra.csv"
df_materias = pd.read_csv(caminho_csv)

print(f"Total de matérias carregadas: {len(df_materias)}")

Total de matérias carregadas: 181


A função do LangChain nesta etapa é puramente mecânica: ele fornece um cortador de texto inteligente. O RecursiveCharacterTextSplitter tenta fatiar o texto primeiro por parágrafos (\n\n), depois por quebras de linha (\n), e depois por pontos finais (.). Isso garante que uma frase importante não seja cortada pela metade, preservando o sentido (a semântica) da notícia.

In [2]:
# chunk_size: Quantidade máxima de caracteres por pedaço. 800 é um bom tamanho para artigos.
# chunk_overlap: Quantidade de caracteres que o final de um chunk repete no início do próximo. 
#                Isso impede que o contexto se perca na transição de um pedaço para o outro.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [3]:
documentos_processados = []

for index, row in df_materias.iterrows():
    # Pula a linha se a coluna de texto estiver vazia
    if pd.isna(row['texto']):
        continue
        
    texto_original = str(row['texto'])
    
    # O LangChain faz a mágica de fatiar o texto respeitando as regras do Passo 2
    chunks = text_splitter.split_text(texto_original)
    
    # Para cada pedaço gerado, criamos um dicionário vinculando o trecho aos seus metadados
    for chunk in chunks:
        documento = {
            "conteudo": chunk,
            "metadados": {
                "titulo": str(row['titulo']),
                "url": str(row['url']),
                "data_publicacao": str(row['ano_mes'])
            }
        }
        documentos_processados.append(documento)

print(f"Total de fragmentos (chunks) gerados: {len(documentos_processados)}")

Total de fragmentos (chunks) gerados: 1553


## Criando o banco vetorial.

* transformar os textos em vetores matemáticos
* armazenar os vetores em um banco vetorial, que será a única fonte de consulta da nossa IA generativa.

In [4]:
# Este bloco substitui o sqlite3 antigo do sistema pelo novo 
import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:



# 1. Transformação de Formato
# O ChromaDB exige que os textos estejam em um formato específico chamado "Document"
docs_langchain = []
for doc in documentos_processados:
    docs_langchain.append(
        Document(
            page_content=doc["conteudo"], 
            metadata=doc["metadados"]
        )
    )

print(f"Preparados {len(docs_langchain)} documentos para o banco vetorial.")

# 2. O Tradutor (Modelo de Embeddings)
# Vamos usar um modelo multilíngue leve e eficiente para rodar no seu ambiente local
print("Baixando/Carregando o modelo de embeddings...")
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Criação e Persistência no ChromaDB
# Escolhemos uma pasta dentro do seu diretório 'data' para salvar o banco fisicamente
diretorio_banco = "../data/chroma_banco_vetorial"

print("Calculando os vetores matemáticos e salvando no disco...")
banco_vetorial = Chroma.from_documents(
    documents=docs_langchain,
    embedding=modelo_embeddings,
    persist_directory=diretorio_banco
)

print(f"Sucesso! Banco vetorial criado e salvo na pasta: {diretorio_banco}")

Preparados 1553 documentos para o banco vetorial.
Baixando/Carregando o modelo de embeddings...


/home/mariana/Projetinhos pessoais/projeto-extra-favelas/.venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Calculando os vetores matemáticos e salvando no disco...
Sucesso! Banco vetorial criado e salvo na pasta: ../data/chroma_banco_vetorial
